### f1: lista dei numeri da 0 a n-1
- Il `for` è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- Senza `Requires(n >= 0)` la verifica fallisce con: *Loop invariant might not hold on entry. Assertion (i <= n) might not hold.* 
    - L'errore non è sulla postcondizione ma sull'ingresso nel ciclo -> Nagini segnala il primo punto della dimostrazione che non regge.

#### Cosa controlla Nagini
* **Statement dimostrato:** per ogni n ≥ 0, len(f1(n)) = n.

1. **Ingresso:** i = 0, l vuota, quindi len(l) = i; i ≤ n vale grazie a n ≥ 0.
2. **Conservazione:** `append` e `i += 1` aumentano entrambi di 1.
3. **Uscita:** guardia falsa (i ≥ n) e invariante (i ≤ n) danno i = n, quindi len(l) = n.

#### Risultato 
* Verification successful


In [ ]:
from typing import List
from nagini_contracts.contracts import *

def f1(n: int) -> List[int]:
    Requires(n >= 0) #precondizione
    Ensures(list_pred(Result())) #permesso sulla lista
    Ensures(len(Result()) == n) #postcondizione , ciò che vogliamo dimostrare
    l: List[int] = []
    i = 0
    while i < n:
        #invariante vera per ogni iterazione
        Invariant(list_pred(l)) # senza questo append fallirebbe per mancanza di permessi
        Invariant(i <= n)
        Invariant(len(l) == i) #punto fondamentale per la dimostrazione della postcondizione
        l.append(i)
        i += 1
    return l

### nFibonacci: primi n numeri della sequenza di Fibonacci
- Il `for` è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- il contratto e le invarianti solo le stesse di f1
- **Importante** 
    - gli invarianti non parlano di a e b. Per la lunghezza i valori sono irrilevanti, e infatti la dimostrazione è identica a quella di f1.
    - Nagini non ha bisogno di sapere niente sul contenuto della lista per dimostrarne la lunghezza.

#### Come ragiona Nagini (induzione sulle iterazioni)
* **Statement dimostrato:** per ogni n ≥ 0, len(nFibonacci(n)) = n.

* i controlli di nagini sono gli stessi che fa per f1

#### Risultato
* Verification successful

In [ ]:
def nFibonacci(n: int) -> List[int]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(len(Result()) == n)
    f: List[int] = []
    a, b = 0, 1
    i = 0
    while i < n:
        Invariant(list_pred(f))
        Invariant(i <= n)
        Invariant(len(f) == i)
        f.append(a)
        a, b = b, a + b #assegnazione simultanea permessa da nagini
        i += 1
    return f

## triangolo: [[0], [0, 1], [0, 1, 2], ...]
- Il `for` esterno è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- Le righe sono oggetti separati dalla lista esterna: serve un permesso quantificato, `Forall(l, lambda r: (list_pred(r), [[len(r)]]))`, per leggerle e modificarle.
- La lunghezza delle righe si specifica con un quantificatore sugli indici, `Forall(int, ...)`, limitato con `Implies` agli indici validi.
- Rispetto a f1 la difficoltà sta nel gestire i permessi su oggetti annidati.

### Cosa controlla Nagini
- **Statement dimostrati:** per ogni n ≥ 0, l'output ha n righe e la riga k ha k+1 elementi.
- Lo statement sulle righe è dimostrato con un `Assert` prima del `return`: vale all'uscita della funzione, ma non è nel contratto, quindi chi chiama `triangolo` non può usarlo.

- **Ciclo interno:** stesso schema di f1, con i+1 al posto di n.

- **Ciclo esterno**
  1. **Ingresso:** i = 0, l vuota; il `Forall` sulle righe vale perché non ci sono righe.
  2. **Conservazione:**
     - le righe già inserite non cambiano: i permessi pieni su `riga` e sulle righe di `l` garantiscono che `riga` non sia una di esse (niente aliasing);
     - la nuova riga ha lunghezza i+1, quindi il `Forall` vale fino a k < i+1;
     - dopo l'`append`, il permesso su `riga` entra nel permesso quantificato su `l`.
  3. **Uscita:** da i ≥ n e i ≤ n segue i = n; quindi n righe, la riga k di lunghezza k+1.

---
### triangolo: percorso di verifica

**Risultato finale:** verificato con Nagini 1.3.1. Per ogni n ≥ 0 l'output ha n righe e la riga k ha k+1 elementi.

#### Tentativi falliti + spiegazione
1. **`Forall(Result(), lambda r: ...)`** → *Encountered Any type*. 
   - `Result()` vuol dire "il valore che la funzione restituisce", ma per il type checker ha tipo `Any`-> Nagini rifiuta variabili di tipo `Any`
2. **`cast(List[List[int]], Result())` come dominio** → *invalid.forall*. 
   - Il problema del tipo Any sparisce, ma si incappa nel problema del tentativo 3
3. **`Forall(l, lambda r: list_pred(r))` senza trigger** → *invalid.forall*
   - Per accettare un `Forall`, Nagini controlla che la parte dopo i due punti sia una condizione vera o falsa, cioè di tipo bool.
   - Con il dominio l, il type checker non riesce a stabilire il tipo di list_pred(r). Non vuol dire che sia sbagliato: semplicemente non lo sa dire.
   - Nel dubbio Nagini rifiuta -> scoperto dal codice sorgente `contract.py` riga 947

5. **`Forall(int, lambda k: Implies(..., list_pred(l[k])))`**, cioè il permesso sulla riga in posizione k, per ogni k → la traduzione passa, ma *insufficient permission* 
   - Nagini accetta la scrittura, ma poi, quando deve leggere la lunghezza di una riga, dice che il permesso non c'è.
6. **Invarianti ripetuti nel ciclo interno e `Assert` intermedi** → nessun effetto.
   - Erano prove per capire dove si perdesse il permesso.
   - Non hanno cambiato nulla, ma hanno mostrato che il problema non era nel ciclo: era nel modo in cui il permesso era scritto.
7. **Aggiornamento da Nagini 1.2.0 a 1.3.1** 
   - Da solo non risolve, ma elimina i warning di Z3 e dimezza i tempi di verifica.

#### Soluzione
- I permessi sulle righe vanno quantificati sugli elementi, nella forma con trigger: `Forall(l, lambda r: (list_pred(r), [[len(r)]]))`
   - per ogni riga r di l, ho il permesso su r; applica questa regola quando compare len(r) -> il trigger
   - Forall vale per infiniti casi, e il solver Z3 non può applicarlo a tutti -> lo applica solo quando incontra il tirgger.
- Nella forma `(asserzione, trigger)` Nagini salta il controllo sul tipo del corpo (`contract.py`, righe 937-943). Il trigger `len(r)` fa scattare il quantificatore proprio dove serve, cioè quando si legge la lunghezza di una riga.
- Le lunghezze delle righe restano quantificate sugli indici: `Forall(int, lambda k: Implies(0 <= k and k < i, len(l[k]) == k + 1))`. 
   - per ogni k tra 0 e i-1, la riga in posizione k ha k+1 elementi
   - Qui serve solo leggere, e il permesso arriva dal quantificatore sugli elementi. (Risolve il problema del tentatovo 4, che tentava di dare i permessi tramite gli indice)
- Gli invarianti esterni sono ripetuti anche nel ciclo interno -> dovrei verificare se sono effettivamente necessari

#### Cosa si impara su Nagini
- **Permessi su oggetti annidati:** si esprimono quantificando sugli elementi della lista, non sugli indici.
- **La documentazione non basta sempre:** la regola che causava *invalid.forall* si è trovata solo leggendo il codice sorgente.
- **Metodo di debug:** isolare le specifiche in funzioni senza corpo, con le specifiche come precondizioni, e tenere ogni esperimento in un file separato, perché basta una forma rifiutata per bloccare la traduzione dell'intero file.
- **La versione è una variabile dell'esperimento:** la 1.3.1 richiede Python 3.12 o successivo e Java installato nell'ambiente.

In [ ]:
def triangolo(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda r: list_pred(r)))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda k: Implies(0 <= k and k < n, len(Result()[k]) == k + 1)))
    l: List[List[int]] = []
    i = 0
    while i < n:
        Invariant(list_pred(l))
        Invariant(Forall(l, lambda r: list_pred(r)))
        Invariant(0 <= i and i <= n)
        Invariant(len(l) == i)
        Invariant(Forall(int, lambda k: Implies(0 <= k and k < i, len(l[k]) == k + 1)))
        riga: List[int] = []
        j = 0
        while j <= i:
            Invariant(list_pred(riga))
            Invariant(0 <= j and j <= i + 1)
            Invariant(len(riga) == j)
            riga.append(j)
            j += 1
        l.append(riga)
        i += 1
    return l

In [ ]:
def matrixI(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda r: list_pred(r)))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda k: Implies(0 <= k and k < n,
                                          len(Result()[k]) == n)))
    M: List[List[int]] = []
    i = 0
    while i < n:
        Invariant(list_pred(M))
        Invariant(Forall(M, lambda r: list_pred(r)))
        Invariant(0 <= i and i <= n)
        Invariant(len(M) == i)
        Invariant(Forall(int, lambda k: Implies(0 <= k and k < i,
                                                len(M[k]) == n)))
        riga: List[int] = []
        j = 0
        while j < n:
            Invariant(list_pred(riga))
            Invariant(0 <= j and j <= n)
            Invariant(len(riga) == j)
            riga.append(1 if i == j else 0)
            j += 1
        M.append(riga)
        i += 1
    return M

In [ ]:
# Funzione pura per esprimere 2^k nelle specifiche
@Pure
def pow2(k: int) -> int:
    Requires(k >= 0)
    return 1 if k == 0 else 2 * pow2(k - 1)

def sottoinsiemi(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda x: list_pred(x)))
    Ensures(len(Result()) == pow2(n))
    vuoto: List[int] = []
    s: List[List[int]] = [vuoto]
    k = 0
    while k < n:
        Invariant(list_pred(s))
        Invariant(Forall(s, lambda x: list_pred(x)))
        Invariant(0 <= k and k <= n)
        Invariant(len(s) == pow2(k))
        new: List[List[int]] = []
        j = 0
        while j < len(s):
            Invariant(list_pred(s))
            Invariant(Forall(s, lambda x: list_pred(x)))
            Invariant(len(s) == pow2(k))
            Invariant(list_pred(new))
            Invariant(Forall(new, lambda x: list_pred(x)))
            Invariant(0 <= j and j <= len(s))
            Invariant(len(new) == j)
            new.append(s[j] + [k])
            j += 1
        s = s + new
        k += 1
    return s